In [ ]:
import pandas as pd
import openai
import re
import time
import os
from google.colab import userdata

# ==========================================
# 1. API & MODEL CONFIGURATION
# ==========================================

try:
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = "YOUR_API_KEY"

client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Llama 3.1 70B
MODEL_NAME = "meta-llama/llama-3.1-70b-instruct"

# ==========================================
# 2. SELECT DATASET (Uncomment ONE at a time)
# ==========================================

#INPUT_FILE, OUTPUT_FILE = "dataset/bangla_med_qa_correct.csv", "llama31_70b_correct.csv"
#INPUT_FILE, OUTPUT_FILE = "dataset/wrong_answers_v1.csv", "llama31_70b_wrong_answers_v1.csv"
#INPUT_FILE, OUTPUT_FILE = "dataset/wrong_answers_v2.csv", "llama31_70b_wrong_answers_v2.csv"
INPUT_FILE, OUTPUT_FILE = "dataset/wrong_answers_v3.csv", "llama31_70b_wrong_answers_v3.csv"

# ==========================================
# 3. EVALUATION & PARSING FUNCTIONS
# ==========================================

def evaluate_pair(question, proposed_answer, retries=3):

    prompt = f"""You are a strict medical accuracy evaluator. Decide whether the provided model answer is correct for the question.

Only reply with a single digit: 1 or 0. No explanation, no punctuation, no extra text.

1 means the answer is factually correct and medically supported.

0 means the answer is incorrect, incomplete, hallucinated, or contradicts medical facts.

Question: {question}

Model answer: {proposed_answer}

Answer now:"""

    for attempt in range(retries):

        try:

            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0,
                top_p=1,
                max_tokens=5,
            )

            return response.choices[0].message.content.strip()

        except Exception as e:

            if attempt < retries - 1:
                print(f"Retry {attempt + 1}...")
                time.sleep(2 * (attempt + 1))
            else:
                return f"ERROR: {str(e)}"


def parse_binary_score(raw_text):

    if raw_text.startswith("ERROR:"):
        return 0

    clean_text = re.sub(r"<.*?>", "", raw_text).strip()

    match = re.search(r"\b(0|1)\b", clean_text)

    if match:
        return int(match.group(1))

    if clean_text == "1":
        return 1

    return 0


# ==========================================
# 4. EXECUTION LOOP
# ==========================================

def run_pipeline():

    if not os.path.exists(INPUT_FILE):
        print(f"❌ File not found: {INPUT_FILE}")
        return

    print("=" * 60)
    print(f"Model   : {MODEL_NAME}")
    print(f"Dataset : {INPUT_FILE}")
    print("=" * 60)

    df = pd.read_csv(INPUT_FILE)

   # Test with only the first 5 rows
    #df = df.head(5)

    raw_responses = []
    parsed_scores = []

    total = len(df)

    for idx, row in df.iterrows():

        question = row["question"]
        answer = row["answer"]

        raw_output = evaluate_pair(question, answer)

        score = parse_binary_score(raw_output)

        raw_responses.append(raw_output)
        parsed_scores.append(score)

        print(
            f"[{idx+1}/{total}] "
            f"Prediction={score} | Raw='{raw_output}'"
        )

        # Small delay to avoid rate limits
        time.sleep(0.3)

    df["model_raw_response"] = raw_responses
    df["isCorrect"] = parsed_scores

    is_wrong_dataset = "wrong" in INPUT_FILE.lower()

    if is_wrong_dataset:
        evaluator_correct_count = (df["isCorrect"] == 0).sum()
    else:
        evaluator_correct_count = (df["isCorrect"] == 1).sum()

    evaluator_accuracy = (
        evaluator_correct_count / len(df)
    ) * 100

    df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n" + "=" * 60)
    print(f"✅ Output saved to: {OUTPUT_FILE}")

    if is_wrong_dataset:
        print(
            f"Evaluator Accuracy (Correctly predicted 0): "
            f"{evaluator_correct_count}/{len(df)} "
            f"({evaluator_accuracy:.2f}%)"
        )
    else:
        print(
            f"Evaluator Accuracy (Correctly predicted 1): "
            f"{evaluator_correct_count}/{len(df)} "
            f"({evaluator_accuracy:.2f}%)"
        )

    print("=" * 60)


if __name__ == "__main__":
    run_pipeline()

In [1]:
#MERGE AND ACCURACY CHECK
import pandas as pd

MODEL_NAME = "Llama 3.1 70B"

# ==============================
# Read files
# ==============================

correct_df = pd.read_csv("llama31_70b_correct.csv")

wrong1 = pd.read_csv("llama31_70b_wrong_answers_v1.csv")
wrong2 = pd.read_csv("llama31_70b_wrong_answers_v2.csv")
wrong3 = pd.read_csv("llama31_70b_wrong_answers_v3.csv")

# ==============================
# Merge wrong datasets
# ==============================

wrong_df = pd.concat(
    [wrong1, wrong2, wrong3],
    ignore_index=True
)

# Save merged wrong dataset
wrong_df.to_csv(
    "llama31_70b_wrong_merged.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ Merged wrong dataset saved as:")
print("llama31_70b_wrong_merged.csv")

# ==============================
# Accuracy
# ==============================

correct_total = len(correct_df)
correct_correct = (correct_df["isCorrect"] == 1).sum()
correct_acc = correct_correct / correct_total * 100

wrong_total = len(wrong_df)
wrong_correct = (wrong_df["isCorrect"] == 0).sum()
wrong_acc = wrong_correct / wrong_total * 100

overall_correct = correct_correct + wrong_correct
overall_total = correct_total + wrong_total
overall_acc = overall_correct / overall_total * 100

summary = pd.DataFrame({
    "Dataset": [
        "Correct",
        "Wrong (Merged)",
        "Overall"
    ],
    "Correct Predictions": [
        correct_correct,
        wrong_correct,
        overall_correct
    ],
    "Total Samples": [
        correct_total,
        wrong_total,
        overall_total
    ],
    "Accuracy (%)": [
        round(correct_acc, 2),
        round(wrong_acc, 2),
        round(overall_acc, 2)
    ]
})

print("\n")
print(summary)

summary.to_csv(
    "evaluation_summary_llama31_70b.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ Evaluation summary saved as:")
print("evaluation_summary_llama31_70b.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'llama31_70b_correct.csv'